In [1]:
from util.imbalance import information_imbalance

/Users/rafael.calsaverini/dev/personal/python/sandbox/information_imbalance/util/imbalance.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  import tqdm.autonotebook as tqdm


In [2]:
import numpy as np
import pandas as pd
import tqdm.autonotebook as tqdm
from sklearn.metrics import pairwise, mean_squared_error
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestRegressor

from sklearn import datasets
from matplotlib import pyplot as plt
from scipy import spatial as sp

In [3]:
def get_performance_pair(df, feature, target, pre_selected=None):
    feature_space = (pre_selected or []) + [feature]
    feat_to_target = information_imbalance(df[feature_space].values, df[[target]].values)
    target_to_feat = information_imbalance(df[[target]].values, df[feature_space].values)
    return pd.Series([feat_to_target, target_to_feat], index=["feat_to_target", "target_to_feat"])
    
def greedy_optimization(df, features, target):
    pre_selected = []
    imbalances = []
    for k in tqdm.trange(len(features)):
        non_selected = list(set(features) - set(pre_selected))
        current_imbalances = pd.DataFrame({
            feature: get_performance_pair(df, feature, target, pre_selected=pre_selected)
            for feature in tqdm.tqdm(non_selected)
        }).T
        best_feature = current_imbalances.sum(axis=1).idxmin()
        pre_selected = pre_selected + [best_feature]
        imbalances = imbalances + [current_imbalances.loc[best_feature]]
    return pre_selected, imbalances

In [4]:
def draw_diagonal(ax):
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
        np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
    ]

    ax.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
    ax.set_aspect('equal')
    ax.set_xlim(lims)
    ax.set_ylim(lims)

In [5]:
data = datasets.fetch_california_housing()
features = data['feature_names']
target = data['target_names'][0]

df = pd.DataFrame(data['data'], columns=features)
df.loc[:, target] = data['target']

In [6]:
df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [7]:
%time feature_order, imbalances = greedy_optimization(df, features, target)

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

0it [00:00, ?it/s]

KeyboardInterrupt: 